# Segmentation of "Credit-Fearful" Households (Net Worth < $2 Million) Using Multi-Feature K-Means, Standardization, Trimmed Variance, and PCA (2019 SCF Data)

## Latihan konsep dengan data dummy

1. Membuat toy_df (income dan age) dengan satu outlier (1.000.000). Variance biasa dihitung dengan .var(): income ≈ 1,79 × 10¹¹, sangat besar karena outlier.

In [1]:
import pandas as pd
from scipy.stats.mstats import trimmed_var

# Create toy data with an outlier
toy_df = pd.DataFrame({
    "income": [50000, 55000, 60000, 52000, 1000000],  # Last value is outlier
    "age": [30, 35, 40, 32, 38]
})

# Regular variance (sensitive to outliers)
regular_var = toy_df.var()
print("Regular variance:")
print(regular_var)

Regular variance:
income    1.789028e+11
age       1.700000e+01
dtype: float64


2. trimmed_var (membuang 10% di tiap ujung) menghasilkan variance income ≈ 1,43 × 10¹¹. Nilainya lebih kecil karena pengaruh outlier berkurang.

In [2]:
# Trimmed variance removes top and bottom 10% before calculating
# limits=(0.1, 0.1) trims 10% from each tail
trimmed_var_result = toy_df.apply(trimmed_var, limits=(0.1, 0.1))
print("\nTrimmed variance (10% from each tail):")
print(trimmed_var_result)


Trimmed variance (10% from each tail):
income    1.431222e+11
age       1.360000e+01
dtype: float64


3. Membuat X_toy (assets dan vehicles) yang skalanya sangat berbeda. Rata-rata dan std dicek sebelum standardisasi.

In [3]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Create toy data with different scales
X_toy = pd.DataFrame({
    "assets": [100000, 200000, 150000, 180000],
    "vehicles": [1, 2, 1, 3]
})

print("Before standardization:")
print(X_toy.agg(["mean", "std"]).round(2))

Before standardization:
         assets  vehicles
mean  157500.00      1.75
std    43493.29      0.96


4. StandardScaler mengubah data sehingga mean ≈ 0 dan std ≈ 1.15 (angka ini muncul karena StandardScaler memakai std populasi, sedangkan pandas memakai std sampel).

In [4]:
# Apply StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_toy)

# Convert back to DataFrame for inspection
X_scaled_df = pd.DataFrame(X_scaled, columns=X_toy.columns)
print("\nAfter standardization:")
print(X_scaled_df.agg(["mean", "std"]).round(2))


After standardization:
      assets  vehicles
mean    0.00     -0.00
std     1.15      1.15


5. make_pipeline(StandardScaler(), KMeans(n_clusters=2)) menggabungkan scaling dan klastering. Hasilnya adalah label [1 0 1 0] dan inertia 1,75.

In [5]:
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans

# Create a pipeline that standardizes then clusters
model = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=2, random_state=42, n_init=10)
)

# Fit the pipeline on toy data
model.fit(X_toy)

# Access the KMeans step to get labels
print("Cluster labels:", model[-1].labels_)
print("Inertia:", model[-1].inertia_)

Cluster labels: [1 0 1 0]
Inertia: 1.7492991589907887


6. Data sintetis 2 kelompok (40 titik). Inertia 10,20 dan silhouette score 0,76 menunjukkan klaster terpisah dengan baik.

In [6]:
from sklearn.metrics import silhouette_score

# Generate slightly larger toy data for meaningful metrics
np.random.seed(42)
X_eval = pd.DataFrame({
    "feature1": np.concatenate([
        np.random.normal(0, 1, 20),
        np.random.normal(5, 1, 20)
    ]),
    "feature2": np.concatenate([
        np.random.normal(0, 1, 20),
        np.random.normal(5, 1, 20)
    ])
})

# Fit a pipeline and extract metrics
eval_model = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=2, random_state=42, n_init=10)
)
eval_model.fit(X_eval)

# Get inertia from the KMeans step
inertia = eval_model[-1].inertia_
print(f"Inertia: {inertia:.2f}")

# Calculate silhouette score (needs original data and labels)
sil_score = silhouette_score(X_eval, eval_model[-1].labels_)
print(f"Silhouette score: {sil_score:.2f}")

Inertia: 10.20
Silhouette score: 0.76


7. PCA(n_components=2) mereduksi 4 fitur menjadi 2 dimensi. Variance yang dijelaskan sekitar 62%.

In [7]:
from sklearn.decomposition import PCA

# Create toy data with 4 features
np.random.seed(42)
X_high_dim = pd.DataFrame({
    "f1": np.random.randn(50),
    "f2": np.random.randn(50),
    "f3": np.random.randn(50),
    "f4": np.random.randn(50)
})

print(f"Original shape: {X_high_dim.shape}")

# Reduce to 2 dimensions
pca = PCA(n_components=2, random_state=42)
X_reduced = pca.fit_transform(X_high_dim)

print(f"Reduced shape: {X_reduced.shape}")
print(f"\nExplained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.2%}")

Original shape: (50, 4)
Reduced shape: (50, 2)

Explained variance ratio: [0.35156876 0.26841   ]
Total variance explained: 62.00%


8. Mengubah hasil PCA menjadi DataFrame PC1 dan PC2.

In [8]:
# Convert to DataFrame for easier handling
X_pca_df = pd.DataFrame(X_reduced, columns=["PC1", "PC2"])
X_pca_df.head()

,PC1,PC2
0,-1.454963,0.515254
1,-0.094405,0.323658
2,-0.344666,0.211783
3,-1.471005,1.152157
4,-0.539874,-0.085761


9. Komentar instalasi plotly dan nbformat (tidak dieksekusi).

In [9]:
# pip install plotly

In [10]:
# pip install nbformat

10. Contoh bar chart horizontal dengan plotly.express.

In [11]:
import plotly.express as px

# Example: horizontal bar chart
categories = pd.Series([300, 500, 200], index=["A", "B", "C"])
fig = px.bar(
    x=categories.values,
    y=categories.index,
    orientation="h",
    title="Example Horizontal Bar Chart"
)
fig.update_layout(xaxis_title="Value", yaxis_title="Category")
fig.show()

11. Contoh scatter plot dengan pewarnaan per grup.

In [12]:
# Example: scatter plot with color grouping
scatter_df = pd.DataFrame({
    "x": [1, 2, 3, 4, 5, 6],
    "y": [2, 3, 1, 5, 4, 6],
    "group": ["A", "A", "A", "B", "B", "B"]
})
fig = px.scatter(
    scatter_df,
    x="x",
    y="y",
    color="group",
    title="Example Scatter Plot with Groups"
)
fig.show()

12. Contoh box plot untuk mendeteksi outlier.

In [13]:
# Example: box plot to check for outliers
box_data = pd.DataFrame({"values": [10, 12, 11, 13, 100, 14, 15]})
fig = px.box(box_data, x="values", title="Example Box Plot")
fig.show()

## Proyek utama dengan data SCF 2019

1. Mengimpor pandas, plotly, trimmed_var, KMeans, PCA, silhouette_score, make_pipeline, dan StandardScaler.

In [14]:
import pandas as pd
import plotly.express as px
from scipy.stats.mstats import trimmed_var
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

2. Fungsi wrangle() membaca CSV lalu memfilter TURNFEAR == 1 dan NETWORTH < 2 juta.

In [15]:
def wrangle(filepath):
    # Load CSV file
    df = pd.read_csv(filepath)
    # Create mask for credit-constrained households with NW < $2M
    mask = (df["TURNFEAR"] == 1) & (df["NETWORTH"] < 2e6)
    # Apply mask
    df = df[mask]
    return df

3. Memuat data. Hasilnya 4.418 baris × 351 kolom.

In [16]:
df = wrangle("data/SCFP2019.csv.gz")
print("DataFrame shape:", df.shape)
df.head()

DataFrame shape: (4418, 351)


,YY1,Y1,WGT,HHSEX,AGE,AGECL,EDUC,EDCL,MARRIED,KIDS,...,NWCAT,INCCAT,ASSETCAT,NINCCAT,NINC2CAT,NWPCTLECAT,INCPCTLECAT,NINCPCTLECAT,INCQRTCAT,NINCQRTCAT
5,2,21,3790.476607,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2
6,2,22,3798.868505,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,3,2,2
7,2,23,3799.468393,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2
8,2,24,3788.076005,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2
9,2,25,3793.066589,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2


4. assert memvalidasi jumlah baris/kolom, bahwa TURNFEAR hanya bernilai 1, dan bahwa NETWORTH maksimum < 2 juta.

In [17]:
assert df.shape[0] > 1000, (
    f"Expected more than 1000 rows, got {df.shape[0]}. "
    "Check your TURNFEAR and NETWORTH filters."
)
assert df.shape[1] > 300, (
    f"Expected more than 300 columns, got {df.shape[1]}. "
    "Make sure you're loading the full dataset."
)
assert df["TURNFEAR"].unique().tolist() == [1], (
    "TURNFEAR should only contain value 1 after filtering."
)
assert df["NETWORTH"].max() < 2e6, (
    f"Max NETWORTH is {df['NETWORTH'].max():,.0f}, expected < 2,000,000."
)
print("Data preparation checkpoint passed!")

Data preparation checkpoint passed!


5. Menghitung variance semua kolom lalu mengambil 10 terbesar (ASSET, NFIN, NETWORTH, dan lain-lain).

In [18]:
# Calculate variance and get top 10
top_ten_var = df.var().sort_values().tail(10)
top_ten_var

PLOAN1      1.140894e+10
ACTBUS      1.251892e+10
BUS         1.256643e+10
KGTOTAL     1.346475e+10
DEBT        1.848252e+10
NHNFIN      2.254163e+10
HOUSES      2.388459e+10
NETWORTH    4.847029e+10
NFIN        5.713939e+10
ASSET       8.303967e+10
dtype: float64

6. Bar chart horizontal dari 10 variance terbesar.

In [19]:
# Create horizontal bar chart
fig = px.bar(
    x=top_ten_var,
    y=top_ten_var.index,
    orientation="h",
    title="SCF: High Variance Features"
)
fig.update_layout(xaxis_title="Variance", yaxis_title="Feature")
fig.show()

7. Box plot NHNFIN (aset non-rumah non-finansial) untuk melihat sebaran dan outlier.

In [20]:
# Create boxplot to examine distribution
fig = px.box(
    data_frame=df,
    x="NHNFIN",
    title="Distribution of Non-home, Non-Financial Assets"
)
fig.update_layout(xaxis_title="Value [$]")
fig.show()

8. Menghitung trimmed variance (10% per sisi) dan mengambil 10 terbesar.

In [21]:
# Calculate trimmed variance (trim 10% from each tail) for all features
# Sort and get the 10 largest, assign to top_ten_trim_var
# Hint: Use df.apply(trimmed_var, limits=(0.1, 0.1))
top_ten_trim_var = (
    df.apply(trimmed_var, limits=(0.1, 0.1)).sort_values().tail(10)
)
top_ten_trim_var

WAGEINC     5.550737e+08
HOMEEQ      7.338377e+08
NH_MORT     1.333125e+09
MRTHEL      1.380468e+09
PLOAN1      1.441968e+09
DEBT        3.089865e+09
NETWORTH    3.099929e+09
HOUSES      4.978660e+09
NFIN        8.456442e+09
ASSET       1.175370e+10
dtype: float64

9. Bar chart dari 10 trimmed variance terbesar.

In [22]:
# Create horizontal bar chart of trimmed variance
fig = px.bar(
    x=top_ten_trim_var,
    y=top_ten_trim_var.index,
    orientation="h",
    title="SCF: High Variance Features"
)
fig.update_layout(xaxis_title="Trimmed Variance", yaxis_title="Feature")
fig.show()

10. Mengambil 5 fitur teratas: DEBT, NETWORTH, HOUSES, NFIN, ASSET sebagai high_var_cols.

In [23]:
# Extract the column names of the top 5 features by trimmed variance
# Convert to a list and assign to high_var_cols
high_var_cols = top_ten_trim_var.tail(5).index.to_list()
high_var_cols

['DEBT', 'NETWORTH', 'HOUSES', 'NFIN', 'ASSET']

11. Membuat matriks fitur X (4.418 × 5).

In [24]:
# Create feature matrix with selected columns
X = df[high_var_cols]
print("X shape:", X.shape)
X.head()

X shape: (4418, 5)


,DEBT,NETWORTH,HOUSES,NFIN,ASSET
5,12200.0,-6710.0,0.0,3900.0,5490.0
6,12600.0,-4710.0,0.0,6300.0,7890.0
7,15300.0,-8115.0,0.0,5600.0,7185.0
8,14100.0,-2510.0,0.0,10000.0,11590.0
9,15400.0,-5715.0,0.0,8100.0,9685.0


12. assert validasi pemilihan fitur.

In [25]:
assert len(high_var_cols) == 5, (
    f"Expected 5 features, got {len(high_var_cols)}."
)
assert all(col in df.columns for col in high_var_cols), (
    "One or more selected columns not found in DataFrame."
)
assert X.shape[1] == 5, (
    f"Feature matrix should have 5 columns, got {X.shape[1]}."
)
assert X.shape[0] == df.shape[0], (
    f"Feature matrix rows ({X.shape[0]}) should match df rows ({df.shape[0]})."
)
print("Feature selection checkpoint passed!")

Feature selection checkpoint passed!


13. X_summary berisi mean dan std tiap fitur. Skalanya berbeda-beda (misalnya mean ASSET ≈ 149 ribu).

In [26]:
# Create X_summary with mean and std for all features in X
# Convert to integers for readability using .astype(int)
X_summary = X.agg(["mean", "std"]).astype(int)
X_summary

,DEBT,NETWORTH,HOUSES,NFIN,ASSET
mean,72701,76387,74530,117330,149089
std,135950,220159,154546,239038,288166


14. StandardScaler diterapkan pada X menjadi X_scaled.

In [27]:
# Instantiate a StandardScaler
ss = StandardScaler()
# Fit and transform X, storing result in X_scaled_data
X_scaled_data = ss.fit_transform(X)
# Create DataFrame X_scaled with same column names as X
X_scaled = pd.DataFrame(X_scaled_data, columns=X.columns)
# Print shape and display first few rows
print("X_scaled shape:", X_scaled.shape)
X_scaled.head()

X_scaled shape: (4418, 5)


,DEBT,NETWORTH,HOUSES,NFIN,ASSET
0,-0.445075,-0.377486,-0.48231,-0.474583,-0.498377
1,-0.442132,-0.368401,-0.48231,-0.464541,-0.490047
2,-0.422270,-0.383868,-0.48231,-0.467470,-0.492494
3,-0.431097,-0.358407,-0.48231,-0.449061,-0.477206
4,-0.421534,-0.372966,-0.48231,-0.457010,-0.483818


15. Verifikasi bahwa mean ≈ 0 dan std ≈ 1 setelah scaling.

In [28]:
# Create X_scaled_summary with mean and std for X_scaled
# Round to verify mean ≈ 0 and std ≈ 1
X_scaled_summary = X_scaled.agg(["mean", "std"]).round(2)
X_scaled_summary

,DEBT,NETWORTH,HOUSES,NFIN,ASSET
mean,0.0,0.0,-0.0,0.0,0.0
std,1.0,1.0,1.0,1.0,1.0


16. assert untuk memvalidasi hasil standardisasi.

In [29]:
means = X_scaled.mean()
stds = X_scaled.std()
assert all(abs(m) < 1e-10 for m in means), (
    f"Means should be ~0 after standardization, got {means.tolist()}"
)
assert all(abs(s - 1) < 0.01 for s in stds), (
    f"Stds should be ~1 after standardization, got {stds.tolist()}"
)
print("Standardization checkpoint passed!")

Standardization checkpoint passed!


17. Loop k = 2 sampai 12: melatih pipeline scaler+KMeans lalu mencatat inertia dan silhouette. Untuk k = 2 hasilnya inertia ≈ 11.028 dan silhouette 0,746.

In [30]:
n_clusters = range(2, 13)
inertia_errors = []
silhouette_scores = []

# Train models and collect metrics
for k in n_clusters:
    model = make_pipeline(
        StandardScaler(),
        KMeans(n_clusters=k, random_state=42)
    )
    model.fit(X)
    inertia_errors.append(model[-1].inertia_)
    silhouette_scores.append(silhouette_score(X, model[-1].labels_))

print("Inertia (first 3):", inertia_errors[:3])
print("Silhouette scores (first 3):", silhouette_scores[:3])

Inertia (first 3): [11028.058082607175, 7190.703769068748, 6233.106346879298]
Silhouette scores (first 3): [0.7464502937083215, 0.7042841259119152, 0.7046977054481021]


18. Plot inertia vs jumlah klaster (mencari titik elbow).

In [31]:
# Create a line plot of inertia_errors vs n_clusters
fig = px.line(
    x=n_clusters,
    y=inertia_errors,
    title="K-Means Model: Inertia vs Number of Clusters"
)
# Set x-axis to "Number of Clusters", y-axis to "Inertia"
fig.update_layout(xaxis_title="Number of Clusters", yaxis_title="Inertia")
fig.show()

19. Plot silhouette score vs jumlah klaster.

In [32]:
# Create a line plot of silhouette_scores vs n_clusters
fig = px.line(
    x=n_clusters,
    y=silhouette_scores,
    title="K-Means Model: Silhouette Score vs Number of Clusters"
)
# Set x-axis to "Number of Clusters", y-axis to "Silhouette Score"
fig.update_layout(
    xaxis_title="Number of Clusters",
    yaxis_title="Silhouette Score"
)
fig.show()

20. assert validasi hasil pemilihan model (11 nilai, inertia menurun).

In [33]:
assert len(inertia_errors) == 11, (
    f"Expected 11 inertia values (k=2 to 12), got {len(inertia_errors)}."
)
assert len(silhouette_scores) == 11, (
    f"Expected 11 silhouette scores, got {len(silhouette_scores)}."
)
assert all(ie > 0 for ie in inertia_errors), (
    "All inertia values should be positive."
)
assert all(-1 <= ss <= 1 for ss in silhouette_scores), (
    "Silhouette scores should be between -1 and 1."
)
assert inertia_errors[0] > inertia_errors[-1], (
    "Inertia should decrease as k increases."
)
print("Model selection checkpoint passed!")

Model selection checkpoint passed!


21. Melatih final_model dengan n_clusters=4.

In [34]:
# Create final_model pipeline with StandardScaler and KMeans
# Use n_clusters=4 and random_state=42
final_model = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=4, random_state=42)
)
# Fit the model on X
final_model.fit(X)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('kmeans', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['DEBT','NETWORTH','HOUSES','NFIN','ASSET']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


22. Mengambil labels_ dari model.

In [35]:
# Extract labels from final_model using named_steps["kmeans"].labels_
labels = final_model.named_steps["kmeans"].labels_
# Print the first 5 labels
print("First 5 labels:", labels[:5])

First 5 labels: [1 1 1 1 1]


23. assert validasi model final (4 klaster, label 0–3).

In [36]:
assert hasattr(final_model, "named_steps"), (
    "final_model should be a pipeline with named_steps."
)
assert "kmeans" in final_model.named_steps, (
    "Pipeline should contain a step named 'kmeans'."
)
assert final_model.named_steps["kmeans"].n_clusters == 4, (
    f"Expected 4 clusters, got {final_model.named_steps['kmeans'].n_clusters}."
)
assert len(labels) == X.shape[0], (
    f"Labels length ({len(labels)}) should match X rows ({X.shape[0]})."
)
assert set(labels) == {0, 1, 2, 3}, (
    f"Expected labels 0-3, got unique values: {set(labels)}."
)
print("Final model checkpoint passed!")

Final model checkpoint passed!


24. X.groupby(labels).mean() menghitung profil rata-rata tiap klaster (xgb).

In [37]:
# Group X by labels and calculate mean for each cluster
# Assign result to xgb (cluster profile DataFrame)
xgb = X.groupby(labels).mean()
xgb

,DEBT,NETWORTH,HOUSES,NFIN,ASSET
0,205111.440049,2.314333e+05,261927.688504,3.512653e+05,4.365447e+05
1,28650.810927,1.176536e+04,14647.482838,2.825256e+04,4.041617e+04
2,669360.967742,5.819507e+05,788306.451613,1.118355e+06,1.251312e+06
3,266576.274510,1.432137e+06,339117.647059,1.295769e+06,1.698713e+06


25. Grouped bar chart profil keuangan per klaster.

In [38]:
# Create grouped bar chart of cluster profiles
fig = px.bar(
    xgb,
    barmode="group",
    title="Mean Household Finances by Cluster"
)
# Set x-axis to "Cluster", y-axis to "Value [$]"
fig.update_layout(xaxis_title="Cluster", yaxis_title="Value [$]")
fig.show()

26. PCA mereduksi X ke 2 komponen (X_pca).

In [39]:
# Instantiate PCA with n_components=2 and random_state=42
pca = PCA(n_components=2, random_state=42)
# Fit and transform X, storing result in X_t
X_t = pca.fit_transform(X)
# Create DataFrame X_pca with columns "PC1" and "PC2"
X_pca = pd.DataFrame(X_t, columns=["PC1", "PC2"])
# Print shape and display first few rows
print("X_pca shape:", X_pca.shape)
X_pca.head()

X_pca shape: (4418, 2)


,PC1,PC2
0,-221525.424530,22052.273003
1,-217775.100722,22851.358068
2,-219519.642175,19023.646333
3,-212195.720367,22957.107039
4,-215540.507551,20259.749306


27. Scatter plot PCA berwarna per klaster.

In [40]:
# Create a scatter plot of X_pca with PC1 on x-axis, PC2 on y-axis
# Color points by cluster labels (convert to string for discrete colors)
fig = px.scatter(
    data_frame=X_pca,
    x="PC1",
    y="PC2",
    color=labels.astype(str),
    title="PCA Representation of Clusters"
)
fig.update_layout(xaxis_title="PC1", yaxis_title="PC2")
fig.show()

28. assert validasi hasil visualisasi.

In [41]:
assert xgb.shape == (4, 5), (
    f"Cluster profile shape should be (4, 5), got {xgb.shape}."
)
assert X_pca.shape == (X.shape[0], 2), (
    f"X_pca shape should be ({X.shape[0]}, 2), got {X_pca.shape}."
)
assert list(X_pca.columns) == ["PC1", "PC2"], (
    f"X_pca columns should be ['PC1', 'PC2'], got {list(X_pca.columns)}."
)
print("Visualization checkpoint passed!")

Visualization checkpoint passed!


Kesimpulan Singkat

- Tujuan: Mengelompokkan 4.418 rumah tangga yang takut kredit dengan net worth di bawah $2 juta berdasarkan 5 fitur keuangan: DEBT, NETWORTH, HOUSES, NFIN, ASSET.
- Pemilihan fitur: Fitur dipilih dengan trimmed variance, bukan variance biasa, supaya outlier tidak mendominasi.
Preprocessing: Standardisasi dengan StandardScaler dalam pipeline, sehingga semua fitur setara bobotnya.
- Pemilihan k: Dari k = 2–12, silhouette tertinggi ada pada k = 2 (0,746). Inertia turun tajam sampai k ≈ 3–4. Model final memakai k = 4 sebagai kompromi antara kerapatan dan jumlah segmen yang bermakna.
- Profil klaster (rata-rata, dalam dolar):
    - Klaster 1: Utang ≈ 28,7 ribu, net worth ≈ 11,8 ribu, aset ≈ 40 ribu. Ini kelompok kecil, keuangan paling lemah (kemungkinan kelompok terbesar, terlihat dari 5 label pertama).
    - Klaster 0: Utang ≈ 205 ribu, net worth ≈ 231 ribu, nilai rumah ≈ 262 ribu. Kelompok menengah dengan kepemilikan rumah.
    - Klaster 2: Utang ≈ 669 ribu, net worth ≈ 582 ribu, aset ≈ 1,25 juta. Kelompok dengan utang dan aset besar (leverage tinggi).
    - Klaster 3: Utang ≈ 267 ribu, net worth ≈ 1,43 juta, aset ≈ 1,70 juta. Kelompok kaya dengan utang relatif kecil.
- Visualisasi: PCA 2 dimensi dipakai untuk melihat sebaran klaster. 

Catatan: 
- PCA dijalankan pada X yang belum diskalakan, jadi PC1 didominasi fitur berskala besar (ASSET, NFIN). 
- Untuk konsistensi dengan klastering, PCA sebaiknya dijalankan pada X_scaled.

Catatan kritis: 
- Jumlah anggota tiap klaster tidak dicetak, sehingga ukuran klaster belum terkonfirmasi. 
- Pipeline pada loop sebenarnya sudah menskalakan data, tetapi silhouette_score dihitung pada X asli (belum diskalakan), sehingga nilainya tidak sepenuhnya sebanding dengan ruang tempat klastering dilakukan.